In [ ]:
import sys
sys.path.append('../')

import torch
import pandas as pd
import optuna
import warnings
from copy import deepcopy
from torch.utils.data import DataLoader
from src.mypackage.data_preparation import prepare_statistical_data
from src.mypackage.torch_dataset import EnergyDataset
from src.mypackage.trainer import Trainer
from src.mypackage.transformer_models import TimeSeriesTransformerModel
from src.mypackage.forecasting import dnn_forecast
from src.mypackage.evaluation import print_metrics, get_true_values
from src.mypackage.visualization import plot_forecast
from src.mypackage.utils import set_seed, SEED

warnings.filterwarnings("ignore")
set_seed(SEED)

In [ ]:
# ========== データ読み込みとデータセット準備 ==========
df = pd.read_csv("../data/raw/PJME_hourly.csv")
_, tmp = prepare_statistical_data(df)
timesteps = tmp["Datetime"].copy()
del tmp
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

SEQ_LEN = 168
PRED_LEN = 24
SHIFT = 24
BATCH_SIZE = 32

# ========== データセット作成 ==========
dataset = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=PRED_LEN, mode="re-train")
retrain_dataset = deepcopy(dataset)
train_dataset = deepcopy(dataset)
train_dataset.mode_switch("train")
valid_dataset = deepcopy(dataset)
valid_dataset.mode_switch("val")
test_dataset = deepcopy(dataset)
test_dataset.mode_switch("test")

# DataLoader作成
retrain_loader = DataLoader(retrain_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Whole dataset size: {len(retrain_dataset)}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(valid_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Input features: {train_dataset[0][0].shape[1]}")

Dataset shape: (145366, 2)
Columns: ['Datetime', 'PJME_MW']
Train dataset size: 4523
Val dataset size: 1131
Test dataset size: 358
Input features: 19


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== ハイパーパラメータ設定 ==========
# 固定
NUM_EPOCHS = 100
INPUT_SIZE = train_dataset[0][0].shape[1]
EARLY_STOPPING_PATIENCE = 10
OUTPUT_SIZE = PRED_LEN

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    hidden_size = 2 ** trial.suggest_int("hidden_size", 4, 9)
    kernel_size = trial.suggest_int("kernel_size", 3, 7)
    num_heads = trial.suggest_int("num_heads", 1, 4)
    dim_feedforward = 2 ** trial.suggest_int("dim_feedforward", 5, 8)
    num_layers = trial.suggest_int("num_layers", 1, 4)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)

    model = TimeSeriesTransformerModel(input_size=INPUT_SIZE, 
                                       output_channels=hidden_size, 
                                       kernel_size=kernel_size,
                                       num_layers=num_layers, 
                                       output_size=OUTPUT_SIZE, 
                                       dropout=dropout,
                                       num_heads=num_heads,
                                       dim_feedforward=dim_feedforward).to(device)

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        checkpoint_dir="../models/transformer_checkpoints"
    )
    history = trainer.train(num_epochs=NUM_EPOCHS, verbose=0)
    best_iteration = trainer.early_stopping.best_iteration
    trial.set_user_attr("best_iteration", best_iteration)
    return min(history['val_loss'])

In [ ]:
# パラメータ探索の実行
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=50)
print(study.best_params)

In [ ]:
# モデルの再学習（1日予測）
model = TimeSeriesTransformerModel(input_size=INPUT_SIZE,
                                   output_channels=2**study.best_params["hidden_size"],
                                   kernel_size=study.best_params["kernel_size"],
                                   output_size=OUTPUT_SIZE,
                                   num_heads=study.best_params["num_heads"],
                                   dim_feedforward=2**study.best_params["dim_feedforward"],
                                   num_layers=study.best_params["num_layers"],
                                   dropout=study.best_params["dropout"]).to(device)

trainer = Trainer(model=model,
                  train_loader=retrain_loader,
                  test_loader=test_loader,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  checkpoint_dir="../models/transformer_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

In [ ]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/transformer_checkpoints/day_best_model.pth")
trainer.save_config("../models/transformer_checkpoints/day_best_model_config.json")


Test Metrics:
  Test Loss (MSE): 0.107888
  Test MAE: 0.233781
  Test RMSE: 0.316225
Checkpoint saved to ..\models\transformer_checkpoints\model_20260510_174309.pt
Configuration saved to ..\models\transformer_checkpoints\training_config_20260510_174309.json


In [ ]:
# 1週間予測用のデータセット
dataset_week = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=7*PRED_LEN, mode="re-train")
test_dataset_week = deepcopy(dataset_week)
test_dataset_week.mode_switch("test")

train_loader_week = DataLoader(dataset_week, batch_size=BATCH_SIZE, shuffle=True)
test_loader_week = DataLoader(test_dataset_week, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# モデルの再学習（1週間予測）
model = TimeSeriesTransformerModel(input_size=INPUT_SIZE,
                                   output_channels=2**study.best_params["hidden_size"],
                                   kernel_size=study.best_params["kernel_size"],
                                   output_size=7*PRED_LEN,
                                   num_heads=study.best_params["num_heads"],
                                   dim_feedforward=2**study.best_params["dim_feedforward"],
                                   num_layers=study.best_params["num_layers"],
                                   dropout=study.best_params["dropout"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_week,
                  test_loader=test_loader_week,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  checkpoint_dir="../models/transformer_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

In [ ]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/transformer_checkpoints/week_best_model.pth")
trainer.save_config("../models/transformer_checkpoints/week_best_model_config.json")

In [ ]:
# 1か月予測用のデータセット
dataset_month = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=30*PRED_LEN, mode="re-train")
test_dataset_month = deepcopy(dataset_month)
test_dataset_month.mode_switch("test")

train_loader_month = DataLoader(dataset_month, batch_size=BATCH_SIZE, shuffle=True)
test_loader_month = DataLoader(test_dataset_month, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# モデルの再学習（1か月予測）
model = TimeSeriesTransformerModel(input_size=INPUT_SIZE,
                                   output_channels=2**study.best_params["hidden_size"],
                                   kernel_size=study.best_params["kernel_size"],
                                   output_size=30*PRED_LEN,
                                   num_heads=study.best_params["num_heads"],
                                   dim_feedforward=2**study.best_params["dim_feedforward"],
                                   num_layers=study.best_params["num_layers"],
                                   dropout=study.best_params["dropout"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_month,
                  test_loader=test_loader_month,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  checkpoint_dir="../models/transformer_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

In [ ]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/transformer_checkpoints/month_best_model.pth")
trainer.save_config("../models/transformer_checkpoints/month_best_model_config.json")

In [ ]:
# 1年予測用のデータセット
dataset_year = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=365*PRED_LEN, mode="re-train")
test_dataset_year = deepcopy(dataset_year)
test_dataset_year.mode_switch("test")

train_loader_year = DataLoader(dataset_year, batch_size=BATCH_SIZE, shuffle=True)
test_loader_year = DataLoader(test_dataset_year, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# モデルの再学習（1年予測）
model = TimeSeriesTransformerModel(input_size=INPUT_SIZE,
                                   output_channels=2**study.best_params["hidden_size"],
                                   kernel_size=study.best_params["kernel_size"],
                                   output_size=365*PRED_LEN,
                                   num_heads=study.best_params["num_heads"],
                                   dim_feedforward=2**study.best_params["dim_feedforward"],
                                   num_layers=study.best_params["num_layers"],
                                   dropout=study.best_params["dropout"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_year,
                  test_loader=test_loader_year,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  checkpoint_dir="../models/transformer_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

In [ ]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/transformer_checkpoints/year_best_model.pth")
trainer.save_config("../models/transformer_checkpoints/year_best_model_config.json")

In [ ]:
# チェックポイントの読み込みと再学習
checkpoint_day = torch.load("../models/transformer_checkpoints/day_best_model.pth", map_location=device)
params_day = checkpoint_day["model_config"]
model_day = TimeSeriesTransformerModel(**params_day).to(device)
model_day.load_state_dict(checkpoint_day["model_state_dict"])

checkpoint_week = torch.load("../models/transformer_checkpoints/week_best_model.pth", map_location=device)
params_week = checkpoint_week["model_config"]
model_week = TimeSeriesTransformerModel(**params_week).to(device)
model_week.load_state_dict(checkpoint_week["model_state_dict"])

checkpoint_month = torch.load("../models/transformer_checkpoints/month_best_model.pth", map_location=device)
params_month = checkpoint_month["model_config"]
model_month = TimeSeriesTransformerModel(**params_month).to(device)
model_month.load_state_dict(checkpoint_month["model_state_dict"])

checkpoint_year = torch.load("../models/transformer_checkpoints/year_best_model.pth", map_location=device)
params_year = checkpoint_year["model_config"]
model_year = TimeSeriesTransformerModel(**params_year).to(device)
model_year.load_state_dict(checkpoint_year["model_state_dict"])

In [ ]:
y_true = get_true_values(test_loader)

y_pred_day = dnn_forecast(model_day, test_loader, dataset, device)
y_pred_week = dnn_forecast(model_week, test_loader_week, dataset, device)[-len(y_true):]
y_pred_month = dnn_forecast(model_month, test_loader_month, dataset, device)[-len(y_true):]
y_pred_year = dnn_forecast(model_year, test_loader_year, dataset, device)

In [ ]:
print_metrics(y_true, y_pred_day, "Transformer Forecast (1 Day)")
print_metrics(y_true, y_pred_week, "Transformer Forecast (1 Week)")
print_metrics(y_true, y_pred_month, "Transformer Forecast (1 Month)")
print_metrics(y_true, y_pred_year, "Transformer Forecast (1 Year)")

In [ ]:
plot_forecast(y_true, y_pred_day, timesteps, "Transformer Forecast (1 Day)")

In [ ]:
plot_forecast(y_true, y_pred_week, timesteps, "Transformer Forecast (1 Week)")

In [ ]:
plot_forecast(y_true, y_pred_month, timesteps, "Transformer Forecast (1 Month)")

In [ ]:
plot_forecast(y_true, y_pred_year, timesteps, "Transformer Forecast (1 Year)")